# AIDE Inference Results

Objective: present the saved AIDE detector outputs for NTIRE and Z-Image-Turbo without re-running model inference by default.

Artifacts preserved by this notebook:
- `data/silver/aide/ntire_scores.csv`
- `data/silver/aide/z_image_turbo_scores.csv`
- `data/silver/aide/ntire_metrics.json`
- `data/silver/aide/plots/`


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import SVG, display
from sklearn.metrics import precision_recall_curve, roc_curve

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent.parent

OUTPUT_DIR = REPO_ROOT / "data" / "silver" / "aide"
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

NTIRE_SCORES = OUTPUT_DIR / "ntire_scores.csv"
ZIT_SCORES = OUTPUT_DIR / "z_image_turbo_scores.csv"
NTIRE_METRICS = OUTPUT_DIR / "ntire_metrics.json"

for path in [NTIRE_SCORES, ZIT_SCORES, NTIRE_METRICS]:
    if not path.exists():
        raise FileNotFoundError(path)

print(f"Repo root: {REPO_ROOT}")
print(f"AIDE artifacts: {OUTPUT_DIR}")

In [ ]:
results_ntire = pd.read_csv(NTIRE_SCORES)
results_zit = pd.read_csv(ZIT_SCORES)
with NTIRE_METRICS.open() as handle:
    metrics_ntire = json.load(handle)

summary_ntire = pd.DataFrame(
    [
        {
            "dataset": "NTIRE",
            "images": len(results_ntire),
            "accuracy": metrics_ntire.get("accuracy"),
            "average_precision": metrics_ntire.get("average_precision"),
            "auc_roc": metrics_ntire.get("auc_roc"),
        }
    ]
)
summary_zit = pd.DataFrame(
    [
        {
            "dataset": "Z-Image-Turbo",
            "images": len(results_zit),
            "mean_score": results_zit["score"].mean(),
            "std_score": results_zit["score"].std(),
            "median_score": results_zit["score"].median(),
            "score_gt_0_5_rate": (results_zit["score"] > 0.5).mean(),
        }
    ]
)

display(summary_ntire.style.format(precision=4))
display(summary_zit.style.format(precision=4))

In [ ]:
y_true = results_ntire["label"].to_numpy()
y_score = results_ntire["score"].to_numpy()

fig, ax = plt.subplots(figsize=(6, 4))
for label, color, name in [(0, "#276FBF", "NTIRE real"), (1, "#C84630", "NTIRE fake")]:
    subset = results_ntire.loc[results_ntire["label"] == label, "score"]
    ax.hist(subset, bins=40, alpha=0.55, label=name, color=color, density=True)
ax.hist(
    results_zit["score"],
    bins=25,
    alpha=0.55,
    label="Z-Image-Turbo fake",
    color="#2A9D8F",
    density=True,
)
ax.axvline(0.5, color="black", linestyle="--", linewidth=0.9)
ax.set_xlabel("Fake probability")
ax.set_ylabel("Density")
ax.set_title("AIDE score distribution")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "score_histogram.svg", format="svg")
plt.close(fig)

fpr, tpr, _ = roc_curve(y_true, y_score)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(
    fpr, tpr, color="#276FBF", linewidth=2, label=f"AUC={metrics_ntire['auc_roc']:.4f}"
)
ax.plot([0, 1], [0, 1], "k--", linewidth=0.9)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("AIDE ROC on NTIRE")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ntire_roc_curve.svg", format="svg")
plt.close(fig)

precision, recall, _ = precision_recall_curve(y_true, y_score)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(recall, precision, color="#276FBF", linewidth=2)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"AIDE PR on NTIRE (AP={metrics_ntire['average_precision']:.4f})")
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ntire_pr_curve.svg", format="svg")
plt.close(fig)

print(f"Saved SVG plots to {PLOTS_DIR}")
display(SVG(filename=PLOTS_DIR / "score_histogram.svg"))

In [ ]:
display(SVG(filename=PLOTS_DIR / "ntire_roc_curve.svg"))
display(SVG(filename=PLOTS_DIR / "ntire_pr_curve.svg"))

In [ ]:
RUN_INFERENCE = False

if RUN_INFERENCE:
    import random

    import numpy as np
    import torch
    import torch.backends.cudnn as cudnn

    from diffguard.aide_inference import AIDEDataset, AIDEInferenceRunner

    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

    checkpoint = REPO_ROOT / "data" / "artifacts" / "checkpoints" / "GenImage_train.pth"
    ntire_labels = pd.read_csv(
        REPO_ROOT / "data" / "bronze" / "ntire" / "test_labels.csv"
    )
    ntire_dir = REPO_ROOT / "data" / "bronze" / "ntire" / "test_images"
    zit_dir = REPO_ROOT / "data" / "bronze" / "z_image_turbo"

    runner = AIDEInferenceRunner(
        checkpoint_path=str(checkpoint),
        device="cuda" if torch.cuda.is_available() else "cpu",
        batch_size=4,
    )
    ntire_samples = [
        (str(ntire_dir / row.image_name), int(row.label))
        for row in ntire_labels.itertuples()
    ]
    zit_samples = [(str(path), 1) for path in sorted(zit_dir.glob("*.png"))]

    runner.run(AIDEDataset(ntire_samples)).to_csv(NTIRE_SCORES, index=False)
    runner.run(AIDEDataset(zit_samples)).to_csv(ZIT_SCORES, index=False)
else:
    print(
        "Inference cell skipped. Set RUN_INFERENCE=True only when regenerating AIDE scores."
    )

## Notes

AIDE generalizes modestly to NTIRE but assigns high fake probability to many Z-Image-Turbo samples. These saved outputs feed the gold-layer comparison notebook.
